In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor,RandomForestClassifier
from sklearn.metrics import mean_squared_error,accuracy_score,r2_score,classification_report,confusion_matrix,roc_auc_score,roc_curve,mean_absolute_error
from sklearn.model_selection import train_test_split,cross_val_score
import warnings
import joblib # Added joblib import
warnings.filterwarnings('ignore')
RANDOM_STATE=42
np.random.seed(RANDOM_STATE)

In [10]:
def generate_indian_dataset(n_samples: int = 5000) -> pd.DataFrame:
    print("="*60)
    print("Generating Indian Dataset")
    print("Based on IRC:37-2018+IRC:82-2015+MORTH 2023")
    current_severity=np.random.choice(
        [1,2,3,4,5],
        size=n_samples,
        p=[0.15,0.25,0.30,0.20,0.10]
    ).astype(float)
    rainfall_season=np.random.choice(
        ["dry","pre_monsoon","monsoon", "post_monsoon"],
        size=n_samples,
        p=[0.35,0.15,0.30,0.20]
    )
    monthly_rainfall = np.where(rainfall_season == "monsoon",
                                 np.random.normal(320, 60, n_samples),
                        np.where(rainfall_season == "pre_monsoon",
                                 np.random.normal(120, 30, n_samples),
                        np.where(rainfall_season == "post_monsoon",
                                 np.random.normal(80, 25, n_samples),
                                 np.random.normal(40, 15, n_samples))))
    monthly_rainfall = np.clip(monthly_rainfall, 5, 600)
    road_type=np.random.choice(
        ["arterial", "collector", "local", "highway"],
        size=n_samples,
        p=[0.30, 0.35, 0.25, 0.10]
    )
    temperature_range = np.random.normal(12, 5, n_samples)
    temperature_range = np.clip(temperature_range, 3, 30)
    crack_intensity = np.random.poisson(lam=2, size=n_samples).astype(float)
    crack_intensity = np.clip(crack_intensity, 0, 15)

    # Generate additional features before use
    vehicles_per_hour = np.random.normal(500, 200, n_samples)
    vehicles_per_hour = np.clip(vehicles_per_hour, 50, 1500).astype(int)
    road_age_years = np.random.normal(7, 4, n_samples)
    road_age_years = np.clip(road_age_years, 1, 25).astype(int)
    drainage_condition = np.random.choice([0,1,2], size=n_samples, p=[0.5, 0.3, 0.2])
    construction_quality = np.random.choice([0,1,2], size=n_samples, p=[0.4, 0.4, 0.2])

    base_rate = 0.02
    rain_mult = np.where(monthly_rainfall > 300, 1.8,
                np.where(monthly_rainfall > 150, 1.3,
                np.where(monthly_rainfall > 60,  1.0, 0.7)))
    traf_mult = np.where(vehicles_per_hour > 1000, 1.5,
                np.where(vehicles_per_hour > 500,  1.2,
                np.where(vehicles_per_hour > 200,  1.0, 0.7)))
    drain_mult = np.where(drainage_condition == 2, 1.15,
                 np.where(drainage_condition == 1, 1.05, 1.0))
    age_mult = np.where(road_age_years > 15, 1.4,
               np.where(road_age_years > 8,  1.2,
               np.where(road_age_years > 3,  1.0, 0.8)))
    const_mult = np.where(construction_quality == 2, 1.2,
                 np.where(construction_quality == 1, 1.1, 1.0))
    weekly_growth = (base_rate * rain_mult * traf_mult *
                     drain_mult * age_mult * const_mult)
    noise = np.random.normal(1.0, 0.20, n_samples)
    weekly_growth = weekly_growth * noise
    weekly_growth = np.clip(weekly_growth, 0.005, 0.15)
    weeks_30 = 30 / 7
    weeks_60 = 60 / 7
    weeks_90 = 90 / 7
    severity_30d = np.minimum(
        current_severity * ((1 + weekly_growth) ** weeks_30), 5.0)
    severity_60d = np.minimum(
        current_severity * ((1 + weekly_growth) ** weeks_60), 5.0)
    severity_90d = np.minimum(
        current_severity * ((1 + weekly_growth) ** weeks_90),5.0)
    will_worsen = (severity_90d - current_severity >= 1.0).astype(int)
    df = pd.DataFrame({
        "current_severity":     current_severity,
        "monthly_rainfall_mm":  monthly_rainfall,
        "vehicles_per_hour":    vehicles_per_hour,
        "road_age_years":       road_age_years,
        "drainage_condition":   drainage_condition,
        "construction_quality": construction_quality,
        "temperature_range":    temperature_range,
        "crack_intensity":      crack_intensity,
        "severity_30d":         severity_30d,
        "severity_60d":         severity_60d,
        "severity_90d":         severity_90d,
        "weekly_growth_rate":   weekly_growth,
        "will_worsen":          will_worsen,
        "road_type":            road_type,
        "rainfall_season":      rainfall_season,
    })
    print(f"\n  Generated {n_samples:,} synthetic road sections")
    print(f"  Will worsen (\u22651 severity in 90d): "
          f"{will_worsen.sum():,} ({will_worsen.mean()*100:.1f}%)")
    print(f"\n  Feature summary:")
    numeric_cols = ["current_severity", "monthly_rainfall_mm",
                    "vehicles_per_hour", "road_age_years",
                    "severity_90d", "weekly_growth_rate"]
    print(df[numeric_cols].describe().round(2).to_string())

    return df

In [4]:
FEATURES = [
    "current_severity",
    "monthly_rainfall_mm",
    "vehicles_per_hour",
    "road_age_years",
    "drainage_condition",
    "construction_quality",
    "temperature_range",
    "crack_intensity",
]
FEATURE_LABELS = {
    "current_severity":     "Current Severity (1–5)",
    "monthly_rainfall_mm":  "Monthly Rainfall (mm)",
    "vehicles_per_hour":    "Traffic Volume (veh/hr)",
    "road_age_years":       "Road Age (years)",
    "drainage_condition":   "Drainage (0=good, 2=poor)",
    "construction_quality": "Construction Quality",
    "temperature_range":    "Temp Range (°C)",
    "crack_intensity":      "Crack Intensity (count)",
}
def train_models(df: pd.DataFrame):
    print("\n" + "=" * 60)
    print("  TRAINING MODELS")
    print("=" * 60)

    X = df[FEATURES]


    print("\n  Model A: Regression (predict severity at 90 days)")
    y_reg = df["severity_90d"]
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y_reg, test_size=0.2, random_state=RANDOM_STATE)

    rf_reg = RandomForestRegressor(
        n_estimators=100, max_depth=10,
        min_samples_leaf=10, random_state=RANDOM_STATE, n_jobs=-1)
    rf_reg.fit(X_tr, y_tr)
    y_pred_reg = rf_reg.predict(X_te)
    mae = mean_absolute_error(y_te, y_pred_reg)
    r2  = r2_score(y_te, y_pred_reg)
    print(f"    MAE (severity error): {mae:.4f}  (target: <0.5)")
    print(f"    R²                  : {r2:.4f}  (target: >0.80)")

    # ── Model B: Classification — will it worsen? ─────────────
    print("\n  Model B: Classification (will it worsen ≥1 in 90 days?)")
    y_cls = df["will_worsen"]
    X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
        X, y_cls, test_size=0.2,
        random_state=RANDOM_STATE, stratify=y_cls)

    rf_cls = RandomForestClassifier(
        n_estimators=100, max_depth=10,
        min_samples_leaf=10, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1)
    rf_cls.fit(X_tr2, y_tr2)
    y_pred_cls = rf_cls.predict(X_te2)
    y_prob_cls = rf_cls.predict_proba(X_te2)[:, 1]
    acc = accuracy_score(y_te2, y_pred_cls)
    roc = roc_auc_score(y_te2, y_prob_cls)
    print(f"    Accuracy : {acc*100:.2f}%")
    print(f"    ROC-AUC  : {roc:.4f}")
    print(f"\n    Classification Report:")
    report = classification_report(y_te2, y_pred_cls,
                                   target_names=["Stable", "Will Worsen"])
    for line in report.split("\n"):
        print(f"      {line}")


    cv_scores = cross_val_score(rf_cls, X, y_cls, cv=5, scoring="roc_auc")
    print(f"\n    5-Fold CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

    return (rf_reg, rf_cls,
            y_te, y_pred_reg,
            y_te2, y_pred_cls, y_prob_cls,
            mae, r2, acc, roc)

In [5]:
def plot_results(df, rf_reg, rf_cls, y_te, y_pred_reg,
                 y_te2, y_pred_cls, y_prob_cls,
                 mae, r2, acc, roc):

    print("\n  Generating charts...")
    fig, axes = plt.subplots(2, 3, figsize=(18, 11))
    fig.suptitle(
        "Stage 6 — Pothole Deterioration Prediction\n"
        "Indian Road Conditions | IRC:37-2018 | Random Forest | Cohort 12",
        fontsize=13, fontweight="bold")

    # ── Plot 1: Feature Importance (Regression) ──────────────
    ax1 = axes[0, 0]
    imp   = rf_reg.feature_importances_
    fidx  = np.argsort(imp)
    flbls = [FEATURE_LABELS[FEATURES[i]] for i in fidx]
    colors = ["#C0392B" if imp[i] > 0.15 else "#E67E22"
              if imp[i] > 0.08 else "#3498DB" for i in fidx]
    ax1.barh(flbls, imp[fidx], color=colors, edgecolor="white")
    ax1.set_xlabel("Importance Score", fontsize=10)
    ax1.set_title("Feature Importance\n(Severity Regression)", fontsize=11, fontweight="bold")
    for i, (imp_val, feat) in enumerate(zip(imp[fidx], flbls)):
        ax1.text(imp_val + 0.002, i, f"{imp_val:.3f}", va="center", fontsize=8)

    # ── Plot 2: Actual vs Predicted Severity ─────────────────
    ax2 = axes[0, 1]
    ax2.scatter(y_te, y_pred_reg, alpha=0.3, s=10,
                color="#2980B9", edgecolors="none")
    ax2.plot([1, 5], [1, 5], "r--", lw=2, label="Perfect prediction")
    ax2.set_xlabel("Actual Severity at 90 days", fontsize=10)
    ax2.set_ylabel("Predicted Severity at 90 days", fontsize=10)
    ax2.set_title(f"Actual vs Predicted\nMAE={mae:.3f}  R²={r2:.3f}", fontsize=11, fontweight="bold")
    ax2.legend(fontsize=9)
    ax2.grid(alpha=0.3)

    # ── Plot 3: Confusion Matrix ──────────────────────────────
    ax3 = axes[0, 2]
    cm = confusion_matrix(y_te2, y_pred_cls)
    sns.heatmap(cm, annot=True, fmt=",", cmap="Reds", ax=ax3,
                xticklabels=["Stable", "Will Worsen"],
                yticklabels=["Stable", "Will Worsen"],
                annot_kws={"size": 12})
    ax3.set_xlabel("Predicted", fontsize=10)
    ax3.set_ylabel("Actual", fontsize=10)
    ax3.set_title(f"Confusion Matrix\nAccuracy: {acc*100:.1f}%", fontsize=11, fontweight="bold")

    # ── Plot 4: ROC Curve ─────────────────────────────────────
    ax4 = axes[1, 0]
    fpr, tpr, _ = roc_curve(y_te2, y_prob_cls)
    ax4.plot(fpr, tpr, "#C0392B", lw=2.5, label=f"ROC (AUC = {roc:.3f})")
    ax4.plot([0,1],[0,1], "k--", lw=1.5, label="Baseline (AUC = 0.5)")
    ax4.fill_between(fpr, tpr, alpha=0.1, color="#C0392B")
    ax4.set_xlabel("False Positive Rate", fontsize=10)
    ax4.set_ylabel("True Positive Rate", fontsize=10)
    ax4.set_title("ROC Curve", fontsize=11, fontweight="bold")
    ax4.legend(fontsize=9)
    ax4.grid(alpha=0.3)

    # ── Plot 5: Severity progression examples ─────────────────
    ax5 = axes[1, 1]
    scenarios = [
        {"label": "Highway, Monsoon\n(Worst case)",
         "sev": 3, "rain": 350, "vph": 900, "age": 12, "drain": 2, "const": 2},
        {"label": "Arterial, Moderate rain\n(Typical Bengaluru)",
         "sev": 2, "rain": 150, "vph": 600, "age": 7,  "drain": 1, "const": 1},
        {"label": "Residential, Dry\n(Best case)",
         "sev": 1, "rain": 40,  "vph": 150, "age": 3,  "drain": 0, "const": 0},
        {"label": "Rural, Post-monsoon\n(Old road)",
         "sev": 3, "rain": 80,  "vph": 400, "age": 18, "drain": 2, "const": 2},
    ]
    colors_scen = ["#C0392B", "#E67E22", "#27AE60", "#8E44AD"]
    days = [0, 30, 60, 90]

    for sc, color in zip(scenarios, colors_scen):
        X_sc = pd.DataFrame([{
            "current_severity":     sc["sev"],
            "monthly_rainfall_mm":  sc["rain"],
            "vehicles_per_hour":    sc["vph"],
            "road_age_years":       sc["age"],
            "drainage_condition":   sc["drain"],
            "construction_quality": sc["const"],
            "temperature_range":    12,
            "crack_intensity":      2,
        }])
        pred_90 = float(rf_reg.predict(X_sc)[0])
        sevs = [sc["sev"],
                min(sc["sev"] + (pred_90 - sc["sev"]) * 0.33, 5),
                min(sc["sev"] + (pred_90 - sc["sev"]) * 0.66, 5),
                min(pred_90, 5)]
        ax5.plot(days, sevs, "o-", color=color, lw=2,
                 label=sc["label"], markersize=5)

    ax5.set_xlabel("Days from detection", fontsize=10)
    ax5.set_ylabel("Predicted Severity (1–5)", fontsize=10)
    ax5.set_title("Severity Progression\nby Road Scenario", fontsize=11, fontweight="bold")
    ax5.set_ylim(0.5, 5.5)
    ax5.set_xticks([0, 30, 60, 90])
    ax5.legend(fontsize=7, loc="upper left")
    ax5.grid(alpha=0.3)
    ax5.axhline(y=4, color="red", linestyle=":", alpha=0.5, label="Critical threshold")

    # ── Plot 6: Model Summary ─────────────────────────────────
    ax6 = axes[1, 2]
    ax6.axis("off")
    summary_lines = [
        ("STAGE 6 — DETERIORATION MODEL", "", True),
        ("", "", False),
        ("Dataset source:",   "Synthetic — IRC:37-2018", False),
        ("Records:",          "5,000 Indian road sections", False),
        ("Features:",         "8 (from image + API + input)", False),
        ("", "", False),
        ("── Regression (Severity @90d) ──", "", True),
        ("MAE:",              f"{mae:.4f}  (target: <0.5)", False),
        ("R²:",               f"{r2:.4f}  (target: >0.80)", False),
        ("", "", False),
        ("── Classification (Will Worsen?) ──", "", True),
        ("Accuracy:",         f"{acc*100:.2f}%", False),
        ("ROC-AUC:",          f"{roc:.4f}  (target: >0.80)", False),
        ("", "", False),
        ("Model saved as:",   "deterioration_model.pkl", False),
        ("", "", False),
        ("Data reference:",   "IRC:37-2018, IRC:82-2015,", False),
        ("",                  "MORTH 2023", False),
    ]
    y_pos = 0.97
    for label, value, bold in summary_lines:
        if label == "":
            y_pos -= 0.04
            continue
        weight = "bold" if bold else "normal"
        color  = "#C0392B" if bold else "#2C3E50"
        ax6.text(0.02, y_pos, label, transform=ax6.transAxes,
                 fontsize=9.5, fontweight=weight, color=color, va="top")
        if value:
            ax6.text(0.52, y_pos, value, transform=ax6.transAxes,
                     fontsize=9.5, color="#2C3E50", va="top")
        y_pos -= 0.055

    plt.tight_layout()
    plt.savefig("stage6_deterioration_india.png", dpi=150, bbox_inches="tight")
    print("  ✅ Chart saved: stage6_deterioration_india.png")
    plt.close()


# ══════════════════════════════════════════════════════════════
# STEP 4 — SAVE MODEL + INFERENCE FUNCTION
# ══════════════════════════════════════════════════════════════

def save_and_test(rf_reg, rf_cls, mae, r2, acc, roc):
    model_data = {
        "regression_model":     rf_reg,
        "classification_model": rf_cls,
        "features":             FEATURES,
        "feature_labels":       FEATURE_LABELS,
        "version":              "1.0",
        "data_source":          "Synthetic — IRC:37-2018 Indian pavement standard",
        "metrics": {
            "regression_mae":  round(mae, 4),
            "regression_r2":   round(r2, 4),
            "classifier_acc":  round(acc, 4),
            "classifier_auc":  round(roc, 4),
        }
    }
    joblib.dump(model_data, "deterioration_model.pkl")

    size_mb = __import__("os").path.getsize("deterioration_model.pkl") / 1024 / 1024
    print(f"\n  ✅ Model saved: deterioration_model.pkl ({size_mb:.1f} MB)")

    # ── Quick inference test ───────────────────────────────────
    print(f"\n  {'─'*55}")
    print(f"  INFERENCE TEST — 3 pothole scenarios")
    print(f"  {'─'*55}")

    test_cases = [
        {
            "name": "Highway pothole — monsoon season",
            "data": {
                "current_severity": 3, "monthly_rainfall_mm": 320,
                "vehicles_per_hour": 900, "road_age_years": 12,
                "drainage_condition": 2, "construction_quality": 1,
                "temperature_range": 10, "crack_intensity": 4,
            }
        },
        {
            "name": "Minor crack — dry residential road",
            "data": {
                "current_severity": 1, "monthly_rainfall_mm": 40,
                "vehicles_per_hour": 150, "road_age_years": 3,
                "drainage_condition": 0, "construction_quality": 0,
                "temperature_range": 10, "crack_intensity": 0,
            }
        },
        {
            "name": "Old rural road — post monsoon",
            "data": {
                "current_severity": 4, "monthly_rainfall_mm": 100,
                "vehicles_per_hour": 500, "road_age_years": 20,
                "drainage_condition": 2, "construction_quality": 2,
                "temperature_range": 15, "crack_intensity": 6,
            }
        },
    ]

    data = joblib.load("deterioration_model.pkl")
    reg  = data["regression_model"]
    cls  = data["classification_model"]
    feat = data["features"]

    for tc in test_cases:
        row    = pd.DataFrame([{f: tc["data"].get(f, 0) for f in feat}])
        sev90  = float(reg.predict(row)[0])
        prob   = float(cls.predict_proba(row)[0][1])
        worsen = "YES ⚠️" if cls.predict(row)[0] == 1 else "NO ✅"

        if prob >= 0.80:   urgency = "🚨 CRITICAL"
        elif prob >= 0.60: urgency = "🔴 HIGH"
        elif prob >= 0.40: urgency = "🟡 MEDIUM"
        else:              urgency = "🟢 LOW"

        print(f"\n  📍 {tc['name']}")
        print(f"     Current severity  : {tc['data']['current_severity']}/5")
        print(f"     Predicted @90 days: {sev90:.2f}/5")
        print(f"     Will worsen?       : {worsen}  (prob: {prob:.1%})")
        print(f"     Urgency           : {urgency}")


In [6]:
if __name__ == "__main__":

    # Generate dataset
    df = generate_indian_dataset(n_samples=5000)

    # Save dataset for reference
    df.to_csv("synthetic_indian_deterioration_data.csv", index=False)
    print(f"\n  Dataset saved: synthetic_indian_deterioration_data.csv")

    # Train models
    (rf_reg, rf_cls,
     y_te, y_pred_reg,
     y_te2, y_pred_cls, y_prob_cls,
     mae, r2, acc, roc) = train_models(df)

    # Visualise
    plot_results(df, rf_reg, rf_cls,
                 y_te, y_pred_reg,
                 y_te2, y_pred_cls, y_prob_cls,
                 mae, r2, acc, roc)

    # Save and test
    save_and_test(rf_reg, rf_cls, mae, r2, acc, roc)

    print("\n" + "=" * 60)
    print("  ALL DONE ✅")
    print("=" * 60)
    print(f"\n  Files created:")
    print(f"    deterioration_model.pkl                — trained model")
    print(f"    synthetic_indian_deterioration_data.csv — training dataset")
    print(f"    stage6_deterioration_india.png         — result charts")

    print("=" * 60)

Generating Indian Dataset
Based on IRC:37-2018+IRC:82-2015+MORTH 2023

  Generated 5,000 synthetic road sections
  Will worsen (≥1 severity in 90d): 2,012 (40.2%)

  Feature summary:
       current_severity  monthly_rainfall_mm  vehicles_per_hour  road_age_years  severity_90d  weekly_growth_rate
count           5000.00              5000.00            5000.00         5000.00       5000.00             5000.00
mean               2.83               142.28             503.38            6.63          3.67                0.03
std                1.19               123.35             197.68            3.75          1.29                0.01
min                1.00                 5.00              50.00            1.00          1.08                0.00
25%                2.00                46.52             368.00            4.00          2.63                0.02
50%                3.00                86.34             505.00            6.00          3.90                0.02
75%                

In [8]:
import shap

# Load the saved model data to access the regression model and features
model_data = joblib.load("deterioration_model.pkl")
rf_reg = model_data["regression_model"]
FEATURES = model_data["features"]

# Define a sample 'row' for SHAP explanation
# Using the 'Highway pothole — monsoon season' scenario from save_and_test as an example
sample_data = {
    "current_severity": 3,
    "monthly_rainfall_mm": 320,
    "vehicles_per_hour": 900,
    "road_age_years": 12,
    "drainage_condition": 2,
    "construction_quality": 1,
    "temperature_range": 10,
    "crack_intensity": 4,
}

row = pd.DataFrame([{f: sample_data.get(f, 0) for f in FEATURES}])

explainer = shap.TreeExplainer(rf_reg)
shap_values = explainer.shap_values(row)

# Feature that contributed most (positively) for this specific pothole
# shap_values[0] is for the regression output, assuming it's a single output model
top_feature_idx = np.argmax(np.abs(shap_values[0]))
formation_reason = FEATURES[top_feature_idx]

print(f"For the sample pothole (current severity {sample_data['current_severity']}/5, monsoon, high traffic):\n")
print(f"  The feature contributing most to its predicted 90-day severity is '{FEATURE_LABELS[formation_reason]}'\n")
print(f"  SHAP values for this sample:\n{pd.Series(shap_values[0], index=[FEATURE_LABELS[f] for f in FEATURES]).round(3)}")

For the sample pothole (current severity 3/5, monsoon, high traffic):

  The feature contributing most to its predicted 90-day severity is 'Current Severity (1–5)'

  SHAP values for this sample:
Current Severity (1–5)       0.621
Monthly Rainfall (mm)        0.430
Traffic Volume (veh/hr)      0.074
Road Age (years)             0.104
Drainage (0=good, 2=poor)    0.067
Construction Quality         0.005
Temp Range (°C)              0.001
Crack Intensity (count)     -0.000
dtype: float64


In [12]:
explainer = shap.TreeExplainer(rf_reg)
model_data["shap_explainer"] = explainer
joblib.dump(model_data, "deterioration_model.pkl")

['deterioration_model.pkl']

In [14]:
import pandas as pd # Ensure pandas is available
import numpy as np # Ensure numpy is available

data = joblib.load("deterioration_model.pkl")
explainer = data["shap_explainer"]
FEATURES = data["features"]
FEATURE_LABELS = data["feature_labels"]

# Define input_row using sample data (consistent with previous SHAP example)
sample_data = {
    "current_severity": 3,
    "monthly_rainfall_mm": 320,
    "vehicles_per_hour": 900,
    "road_age_years": 12,
    "drainage_condition": 2,
    "construction_quality": 1,
    "temperature_range": 10,
    "crack_intensity": 4,
}
input_row = pd.DataFrame([{f: sample_data.get(f, 0) for f in FEATURES}])

shap_vals = explainer.shap_values(input_row)
top_reason = FEATURES[np.argmax(np.abs(shap_vals[0]))]

print(f"For the sample pothole (current severity {sample_data['current_severity']}/5, monsoon, high traffic):\n")
print(f"  The feature contributing most to its predicted 90-day severity is '{FEATURE_LABELS[top_reason]}'")
print(f"  SHAP values for this sample:\n{pd.Series(shap_vals[0], index=[FEATURE_LABELS[f] for f in FEATURES]).round(3)}")

For the sample pothole (current severity 3/5, monsoon, high traffic):

  The feature contributing most to its predicted 90-day severity is 'Current Severity (1–5)'
  SHAP values for this sample:
Current Severity (1–5)       0.621
Monthly Rainfall (mm)        0.430
Traffic Volume (veh/hr)      0.074
Road Age (years)             0.104
Drainage (0=good, 2=poor)    0.067
Construction Quality         0.005
Temp Range (°C)              0.001
Crack Intensity (count)     -0.000
dtype: float64


In [15]:
def get_formation_reason(input_dict):
    reasons = []
    if input_dict["monthly_rainfall_mm"] > 200:
        reasons.append(("Heavy Rainfall", input_dict["monthly_rainfall_mm"] / 600))
    if input_dict["drainage_condition"] >= 2:
        reasons.append(("Poor Drainage", 0.5))
    if input_dict["road_age_years"] > 12:
        reasons.append(("Road Age", input_dict["road_age_years"] / 25))
    if input_dict["vehicles_per_hour"] > 800:
        reasons.append(("High Traffic", input_dict["vehicles_per_hour"] / 1500))
    if input_dict["crack_intensity"] > 4:
        reasons.append(("Existing Cracks", input_dict["crack_intensity"] / 15))

    if not reasons:
        return "General Wear"
    return max(reasons, key=lambda x: x[1])[0]